# Insurance Policy Lifecycle — Exploratory Data Analysis
**Layers covered:** Raw CSV files only (pre-Bronze)

Paths inside the container: `/home/jovyan/work/`

---
## 1. Imports & SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

spark = SparkSession.builder \
    .appName("InsuranceEDA") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Spark version: 3.5.0


---
## 2. Load Raw CSV Files

In [2]:
RAW_PATH = "/home/jovyan/work/data/raw"

df_11_14 = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{RAW_PATH}/motor_data11-14lats.csv")

df_14_18 = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{RAW_PATH}/motor_data14-2018.csv")

print(f"df_11_14 row count : {df_11_14.count():,}")
print(f"df_14_18 row count : {df_14_18.count():,}")

df_11_14 row count : 293,537
df_14_18 row count : 508,500


---
## 3. Schema Inspection

In [3]:
print("=== motor_data11-14lats ===")
df_11_14.printSchema()

print("=== motor_data14-2018 ===")
df_14_18.printSchema()

=== motor_data11-14lats ===
root
 |-- SEX: integer (nullable = true)
 |-- INSR_BEGIN: string (nullable = true)
 |-- INSR_END: string (nullable = true)
 |-- EFFECTIVE_YR: string (nullable = true)
 |-- INSR_TYPE: integer (nullable = true)
 |-- INSURED_VALUE: double (nullable = true)
 |-- PREMIUM: double (nullable = true)
 |-- OBJECT_ID: long (nullable = true)
 |-- PROD_YEAR: integer (nullable = true)
 |-- SEATS_NUM: integer (nullable = true)
 |-- CARRYING_CAPACITY: double (nullable = true)
 |-- TYPE_VEHICLE: string (nullable = true)
 |-- CCM_TON: double (nullable = true)
 |-- MAKE: string (nullable = true)
 |-- USAGE: string (nullable = true)
 |-- CLAIM_PAID: double (nullable = true)

=== motor_data14-2018 ===
root
 |-- SEX: string (nullable = true)
 |-- INSR_BEGIN: string (nullable = true)
 |-- INSR_END: string (nullable = true)
 |-- EFFECTIVE_YR: string (nullable = true)
 |-- INSR_TYPE: integer (nullable = true)
 |-- INSURED_VALUE: double (nullable = true)
 |-- PREMIUM: double (nullabl

In [4]:
print("=== Sample rows: motor_data11-14lats ===")
df_11_14.show(5, truncate=False)

print("=== Sample rows: motor_data14-2018 ===")
df_14_18.show(5, truncate=False)

=== Sample rows: motor_data11-14lats ===
+---+----------+---------+------------+---------+-------------+--------+----------+---------+---------+-----------------+------------+-------+------+---------+----------+
|SEX|INSR_BEGIN|INSR_END |EFFECTIVE_YR|INSR_TYPE|INSURED_VALUE|PREMIUM |OBJECT_ID |PROD_YEAR|SEATS_NUM|CARRYING_CAPACITY|TYPE_VEHICLE|CCM_TON|MAKE  |USAGE    |CLAIM_PAID|
+---+----------+---------+------------+---------+-------------+--------+----------+---------+---------+-----------------+------------+-------+------+---------+----------+
|0  |08-AUG-13 |07-AUG-14|08          |1202     |519755.22    |7209.14 |5000029885|2007     |4        |6.0              |Pick-up     |3153.0 |NISSAN|Own Goods|NULL      |
|0  |08-AUG-12 |07-AUG-13|08          |1202     |519755.22    |7203.89 |5000029885|2007     |4        |6.0              |Pick-up     |3153.0 |NISSAN|Own Goods|NULL      |
|0  |08-AUG-11 |07-AUG-12|08          |1202     |519755.22    |7045.804|5000029885|2007     |4        |6

In [5]:
# Verify column names match between the two files
cols_11_14 = set(df_11_14.columns)
print(cols_11_14)
cols_14_18 = set(df_14_18.columns)
print(cols_14_18)
print("Columns only in 11-14 :", cols_11_14 - cols_14_18)
print("Columns only in 14-18 :", cols_14_18 - cols_11_14)
print("Shared columns        :", len(cols_11_14 & cols_14_18))

{'PROD_YEAR', 'MAKE', 'CLAIM_PAID', 'OBJECT_ID', 'EFFECTIVE_YR', 'PREMIUM', 'INSR_BEGIN', 'TYPE_VEHICLE', 'INSR_TYPE', 'INSR_END', 'INSURED_VALUE', 'CARRYING_CAPACITY', 'CCM_TON', 'SEATS_NUM', 'USAGE', 'SEX'}
{'PROD_YEAR', 'MAKE', 'CLAIM_PAID', 'OBJECT_ID', 'EFFECTIVE_YR', 'PREMIUM', 'INSR_BEGIN', 'TYPE_VEHICLE', 'INSR_TYPE', 'INSR_END', 'INSURED_VALUE', 'CARRYING_CAPACITY', 'CCM_TON', 'SEATS_NUM', 'USAGE', 'SEX'}
Columns only in 11-14 : set()
Columns only in 14-18 : set()
Shared columns        : 16


---
## 4. EDA — Exploratory Data Analysis
> Auto-generated from the data below.

In [6]:
# Union both datasets for combined EDA
df = df_11_14.unionByName(df_14_18)
print(f"Combined row count: {df.count():,}")

Combined row count: 802,037


In [7]:
# Descriptive statistics
df.describe().show(truncate=False)

+-------+------------------+----------+---------+------------------+-------------------+------------------+-----------------+-------------------+------------------+------------------+------------------+------------+------------------+--------------+---------------------+------------------+
|summary|SEX               |INSR_BEGIN|INSR_END |EFFECTIVE_YR      |INSR_TYPE          |INSURED_VALUE     |PREMIUM          |OBJECT_ID          |PROD_YEAR         |SEATS_NUM         |CARRYING_CAPACITY |TYPE_VEHICLE|CCM_TON           |MAKE          |USAGE                |CLAIM_PAID        |
+-------+------------------+----------+---------+------------------+-------------------+------------------+-----------------+-------------------+------------------+------------------+------------------+------------+------------------+--------------+---------------------+------------------+
|count  |802037            |802036    |802036   |802032            |802036             |802036            |802015           |80

In [8]:
# ── Categorical distributions ────────────────────────────────────────────
print('=== TYPE_VEHICLE distribution ===')
df.groupBy('TYPE_VEHICLE').count().orderBy(F.desc('count')).show(truncate=False)

print('=== USAGE distribution ===')
df.groupBy('USAGE').count().orderBy(F.desc('count')).show(truncate=False)

print('=== INSR_TYPE distinct values ===')
df.groupBy('INSR_TYPE').count().orderBy(F.desc('count')).show(truncate=False)

print('=== MAKE top 20 ===')
df.groupBy('MAKE').count().orderBy(F.desc('count')).show(20, truncate=False)

# ── Date range ───────────────────────────────────────────────────────────
print('=== Date range (INSR_BEGIN / INSR_END) ===')
df.select(
    F.min('INSR_BEGIN').alias('min_INSR_BEGIN'),
    F.max('INSR_BEGIN').alias('max_INSR_BEGIN'),
    F.min('INSR_END').alias('min_INSR_END'),
    F.max('INSR_END').alias('max_INSR_END')
).show(truncate=False)

# ── Numeric percentiles (3 separate tables) ──────────────────────────────
pct = [0.0, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0]
labels = ['min', 'p25', 'median', 'p75', 'p95', 'p99', 'max']

def show_percentiles(col_name):
    row = df.select(F.percentile_approx(col_name, pct).alias('vals')).collect()[0]['vals']
    pdf = __import__('pandas').DataFrame({'Percentile': labels, col_name: row})
    print(f'=== {col_name} percentiles ===')
    print(pdf.to_string(index=False))
    print()

show_percentiles('PREMIUM')
show_percentiles('INSURED_VALUE')
show_percentiles('CCM_TON')

=== TYPE_VEHICLE distribution ===
+-------------------------+------+
|TYPE_VEHICLE             |count |
+-------------------------+------+
|Truck                    |151088|
|Pick-up                  |144874|
|Motor-cycle              |143186|
|Automobile               |125993|
|Bus                      |106015|
|Station Wagones          |60785 |
|Trailers and semitrailers|35954 |
|Special construction     |12077 |
|Tractor                  |11414 |
|Tanker                   |10632 |
|Trade plates             |18    |
|NULL                     |1     |
+-------------------------+------+

=== USAGE distribution ===
+----------------------+------+
|USAGE                 |count |
+----------------------+------+
|Own Goods             |219570|
|Private               |205102|
|General Cartage       |125502|
|Fare Paying Passengers|118236|
|Own service           |50521 |
|Taxi                  |46762 |
|Others                |8962  |
|Agricultural Own Farm |7912  |
|Special Construction  |69

---
## 5. Data Quality Analysis
> Auto-generated — all checks feed the DQ Summary table in Section 6.

In [9]:
# Null counts per column
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])
null_counts.show(truncate=False)

+---+----------+--------+------------+---------+-------------+-------+---------+---------+---------+-----------------+------------+-------+----+-----+----------+
|SEX|INSR_BEGIN|INSR_END|EFFECTIVE_YR|INSR_TYPE|INSURED_VALUE|PREMIUM|OBJECT_ID|PROD_YEAR|SEATS_NUM|CARRYING_CAPACITY|TYPE_VEHICLE|CCM_TON|MAKE|USAGE|CLAIM_PAID|
+---+----------+--------+------------+---------+-------------+-------+---------+---------+---------+-----------------+------------+-------+----+-----+----------+
|0  |1         |1       |5           |1        |1            |22     |1        |170      |236      |198200           |1           |9      |1   |2    |741892    |
+---+----------+--------+------------+---------+-------------+-------+---------+---------+---------+-----------------+------------+-------+----+-----+----------+



In [10]:
# ── Collect all DQ findings into a list — feeds DQ Summary (Section 6) ──
total_rows = df.count()
dq_findings = []

# 1. Completeness: null / empty counts per column
null_row = df.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c).cast('string') == ''), c)).alias(c)
    for c in df.columns
]).collect()[0]
for col_name in df.columns:
    n = int(null_row[col_name])
    if n > 0:
        action = 'keep (expected: no-claim rows)' if col_name == 'CLAIM_PAID' else 'investigate'
        dq_findings.append({
            'Column': col_name, 'DQ Dimension': 'Completeness',
            'Issue': f'Null / empty: {n:,}', 'Row Count': n,
            '% of Total': round(100 * n / total_rows, 2), 'Action': action})

# 2. Validity: unparseable dates (expect format dd-MMM-yy)
for date_col in ['INSR_BEGIN', 'INSR_END']:
    bad = df.filter(
        F.to_date(F.col(date_col), 'dd-MMM-yy').isNull() & F.col(date_col).isNotNull()
    ).count()
    if bad > 0:
        dq_findings.append({
            'Column': date_col, 'DQ Dimension': 'Validity',
            'Issue': f'Cannot parse as dd-MMM-yy: {bad:,}', 'Row Count': bad,
            '% of Total': round(100 * bad / total_rows, 2), 'Action': 'drop row'})

# 3. Consistency: INSR_END before INSR_BEGIN
bad_range = df.filter(
    F.to_date(F.col('INSR_END'), 'dd-MMM-yy') < F.to_date(F.col('INSR_BEGIN'), 'dd-MMM-yy')
).count()
if bad_range > 0:
    dq_findings.append({
        'Column': 'INSR_BEGIN / INSR_END', 'DQ Dimension': 'Consistency',
        'Issue': f'INSR_END < INSR_BEGIN: {bad_range:,}', 'Row Count': bad_range,
        '% of Total': round(100 * bad_range / total_rows, 2), 'Action': 'drop row'})

# 4. Validity: PREMIUM <= 0
bad_prem = df.filter(F.col('PREMIUM') <= 0).count()
if bad_prem > 0:
    dq_findings.append({
        'Column': 'PREMIUM', 'DQ Dimension': 'Validity',
        'Issue': f'Zero or negative: {bad_prem:,}', 'Row Count': bad_prem,
        '% of Total': round(100 * bad_prem / total_rows, 2), 'Action': 'drop row'})

# 5. Validity: INSURED_VALUE <= 0
bad_iv = df.filter(F.col('INSURED_VALUE') <= 0).count()
if bad_iv > 0:
    dq_findings.append({
        'Column': 'INSURED_VALUE', 'DQ Dimension': 'Validity',
        'Issue': f'Zero or negative: {bad_iv:,}', 'Row Count': bad_iv,
        '% of Total': round(100 * bad_iv / total_rows, 2), 'Action': 'drop row'})

# 6. Validity: SEX not in {0, 1}
bad_sex = df.filter(~F.col('SEX').isin([0, 1])).count()
if bad_sex > 0:
    dq_findings.append({
        'Column': 'SEX', 'DQ Dimension': 'Validity',
        'Issue': f'Value not in {{0,1}}: {bad_sex:,}', 'Row Count': bad_sex,
        '% of Total': round(100 * bad_sex / total_rows, 2), 'Action': 'flag'})

# 7. Uniqueness: duplicate (OBJECT_ID + INSR_BEGIN + INSR_END)
dup_groups = df.groupBy('OBJECT_ID', 'INSR_BEGIN', 'INSR_END').count() \
               .filter(F.col('count') > 1).count()
if dup_groups > 0:
    dq_findings.append({
        'Column': 'OBJECT_ID + INSR_BEGIN + INSR_END', 'DQ Dimension': 'Uniqueness',
        'Issue': f'Duplicate policy keys ({dup_groups:,} groups)', 'Row Count': dup_groups,
        '% of Total': round(100 * dup_groups / total_rows, 2), 'Action': 'dropDuplicates'})

print(f'Total DQ issues found: {len(dq_findings)}')

Total DQ issues found: 19


---
## 6. DQ Summary Table
> Auto-generated from DQ checks in Section 5.

In [11]:
dq_summary = pd.DataFrame(dq_findings, columns=[
    'Column', 'DQ Dimension', 'Issue', 'Row Count', '% of Total', 'Action'
])
dq_summary.style.set_properties(**{'text-align': 'left'}).hide(axis='index')

Column,DQ Dimension,Issue,Row Count,% of Total,Action
INSR_BEGIN,Completeness,Null / empty: 1,1,0.000000,investigate
INSR_END,Completeness,Null / empty: 1,1,0.000000,investigate
EFFECTIVE_YR,Completeness,Null / empty: 5,5,0.000000,investigate
INSR_TYPE,Completeness,Null / empty: 1,1,0.000000,investigate
INSURED_VALUE,Completeness,Null / empty: 1,1,0.000000,investigate
PREMIUM,Completeness,Null / empty: 22,22,0.000000,investigate
OBJECT_ID,Completeness,Null / empty: 1,1,0.000000,investigate
PROD_YEAR,Completeness,Null / empty: 170,170,0.020000,investigate
SEATS_NUM,Completeness,Null / empty: 236,236,0.030000,investigate
CARRYING_CAPACITY,Completeness,"Null / empty: 198,200",198200,24.710000,investigate


---
## 7. Cleaning Rules Table
> Auto-generated from domain knowledge + DQ findings. These rules drive `bronze_to_silver.py`.

In [12]:
# ── Fixed rules (always apply based on domain knowledge) ────────────────
fixed_rules = [
    {'Column': 'INSR_BEGIN',          'Rule': 'Parse string to DateType',           'Condition': 'Always',                            'Action': "to_date(col,'dd-MMM-yy'); drop if null", 'Output Field': 'insr_begin'},
    {'Column': 'INSR_END',            'Rule': 'Parse string to DateType',           'Condition': 'Always',                            'Action': "to_date(col,'dd-MMM-yy'); drop if null", 'Output Field': 'insr_end'},
    {'Column': 'INSR_BEGIN/INSR_END', 'Rule': 'Remove invalid date ranges',         'Condition': 'insr_end < insr_begin',             'Action': 'drop row',                              'Output Field': '-'},
    {'Column': 'EFFECTIVE_YR',        'Rule': 'Convert 2-digit to 4-digit year',    'Condition': 'Always (0-99 becomes 2000-2099)',   'Action': 'cast int + 2000',                       'Output Field': 'effective_year'},
    {'Column': 'PREMIUM',             'Rule': 'Remove zero / negative premiums',    'Condition': 'PREMIUM <= 0',                      'Action': 'drop row',                              'Output Field': 'premium'},
    {'Column': 'INSURED_VALUE',       'Rule': 'Remove zero / negative values',      'Condition': 'INSURED_VALUE <= 0',                'Action': 'drop row',                              'Output Field': 'insured_value'},
    {'Column': 'CLAIM_PAID',          'Rule': 'Fill nulls with 0.0',               'Condition': 'CLAIM_PAID IS NULL',                'Action': 'fillna(0.0)',                           'Output Field': 'claim_paid'},
    {'Column': 'PROD_YEAR',           'Rule': 'Derive vehicle_age at policy start', 'Condition': 'Always',                            'Action': 'year(insr_begin) - PROD_YEAR',           'Output Field': 'vehicle_age'},
    {'Column': 'PREMIUM',             'Rule': 'Derive premium_segment',             'Condition': 'Always (33rd/66th pct thresholds)', 'Action': 'Low / Medium / High buckets',           'Output Field': 'premium_segment'},
    {'Column': 'INSURED_VALUE',       'Rule': 'Derive value_segment',               'Condition': 'Always (33rd/66th pct thresholds)', 'Action': 'Low / Medium / High buckets',           'Output Field': 'value_segment'},
    {'Column': 'All columns',         'Rule': 'Rename to snake_case lowercase',     'Condition': 'Always',                            'Action': 'col.lower()',                           'Output Field': 'see data dictionary'},
]

# ── Dynamic rules derived from DQ findings ───────────────────────────────
dynamic_rules = []
for f in dq_findings:
    if f['DQ Dimension'] == 'Uniqueness':
        dynamic_rules.append({
            'Column': f['Column'], 'Rule': 'Remove duplicate policy records',
            'Condition': f['Issue'],
            'Action': "dropDuplicates(['OBJECT_ID','INSR_BEGIN','INSR_END'])",
            'Output Field': '-'})
    elif f['DQ Dimension'] == 'Validity' and f['Action'] == 'flag':
        dynamic_rules.append({
            'Column': f['Column'], 'Rule': 'Flag anomalous values',
            'Condition': f['Issue'],
            'Action': 'add boolean flag column',
            'Output Field': f['Column'].lower() + '_flag'})

cleaning_rules = pd.DataFrame(fixed_rules + dynamic_rules)
cleaning_rules.style.set_properties(**{'text-align': 'left'}).hide(axis='index')

Column,Rule,Condition,Action,Output Field
INSR_BEGIN,Parse string to DateType,Always,"to_date(col,'dd-MMM-yy'); drop if null",insr_begin
INSR_END,Parse string to DateType,Always,"to_date(col,'dd-MMM-yy'); drop if null",insr_end
INSR_BEGIN/INSR_END,Remove invalid date ranges,insr_end < insr_begin,drop row,-
EFFECTIVE_YR,Convert 2-digit to 4-digit year,Always (0-99 becomes 2000-2099),cast int + 2000,effective_year
PREMIUM,Remove zero / negative premiums,PREMIUM <= 0,drop row,premium
INSURED_VALUE,Remove zero / negative values,INSURED_VALUE <= 0,drop row,insured_value
CLAIM_PAID,Fill nulls with 0.0,CLAIM_PAID IS NULL,fillna(0.0),claim_paid
PROD_YEAR,Derive vehicle_age at policy start,Always,year(insr_begin) - PROD_YEAR,vehicle_age
PREMIUM,Derive premium_segment,Always (33rd/66th pct thresholds),Low / Medium / High buckets,premium_segment
INSURED_VALUE,Derive value_segment,Always (33rd/66th pct thresholds),Low / Medium / High buckets,value_segment
